In [1]:
import pandas as pd
import requests
import io
from sqlalchemy import create_engine
# --- Supabase Database connection details ---
DB_HOST = "db.wluxnhuyfumkannjvgiu.supabase.co"
DB_PORT = "5432"
DB_NAME = "postgres"
DB_USER = "postgres"
DB_PASS = "BlueISland$$87"   # ← replace this

In [2]:
table_name = 'NHL_Gamelog'
url = 'https://moneypuck.com/moneypuck/playerData/careers/gameByGame/all_teams.csv'

In [3]:
 #Step 1: Download CSV Data
try:
    response = requests.get(url)
    response.raise_for_status()
    file_object = io.StringIO(response.content.decode('utf-8'))
    data = pd.read_csv(file_object)
    print("CSV data loaded successfully.")
except requests.exceptions.RequestException as e:
    print(f"Error fetching CSV: {e}")
    exit()

CSV data loaded successfully.


In [4]:
# Step 2: Standardize gameId in the DataFrame
if 'gameId' in data.columns:
    data['gameId'] = data['gameId'].astype(str).str.strip().str.lower()
else:
    print("Column 'gameId' not found in the data.")
    exit()

In [5]:
 #Step 3: Clean up team abbreviations
team_abbreviation_mapping = {
    'L.A': 'LAK',
    'N.J': 'NJD',
    'S.J': 'SJS',
    'T.B': 'TBL'
}

team_columns = ['team', 'playerTeam', 'opposingTeam']

for col in team_columns:
    if col in data.columns:
        data[col] = data[col].replace(team_abbreviation_mapping)
    else:
        print(f"Column '{col}' not found in the data.")

In [6]:
#Step 4: Connect to Supabase (PostgreSQL)
connection_string = (
    f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
    f"?sslmode=require"
)
try:
    engine = create_engine(connection_string)
    print("Connected to Supabase successfully.")
except Exception as e:
    print(f"Error connecting to Supabase: {e}")
    exit()

Connected to Supabase successfully.


In [7]:
 #Step 5: Deduplicate by gameId and append new data
try:
    existing_game_ids = pd.read_sql(f'SELECT DISTINCT "gameId" FROM "{table_name}"', con=engine)
    existing_game_ids['gameId'] = existing_game_ids['gameId'].astype(str).str.strip().str.lower()

    new_data = data[~data['gameId'].isin(existing_game_ids['gameId'])]
    print(f"New games identified: {len(new_data['gameId'].unique())}")

    if not new_data.empty:
        new_data.to_sql(name=table_name, con=engine, if_exists='append', index=False)
        print(f"Data appended to table '{table_name}' successfully.")
    else:
        print("No new games to append.")

except Exception as e:
    print(f"Error during deduplication or data upload: {e}")

New games identified: 56
Data appended to table 'NHL_Gamelog' successfully.
